# DreamerV3 + Neural Maps on Kaggle
* Full Reinforcement Learning implementation from scratch.
* Runs on NVIDIA GPUs natively.

In [ ]:
!pip install --no-index --find-links /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels arc-agi python-dotenv
!pip install torch torchvision torchaudio

In [ ]:
!mkdir -p models/dreamer

In [ ]:
%%writefile models/dreamer/__init__.py
# Init file

In [ ]:
%%writefile models/dreamer/symlog.py
import torch

def symlog(x: torch.Tensor) -> torch.Tensor:
    """
    Symmetric logarithmic transformation.
    Compresses large magnitudes while remaining linear near zero.
    Used in DreamerV3 for rewards, values, and observations.
    """
    return torch.sign(x) * torch.log(1 + torch.abs(x))

def symexp(x: torch.Tensor) -> torch.Tensor:
    """
    Inverse of the symmetric logarithmic transformation.
    """
    return torch.sign(x) * (torch.exp(torch.abs(x)) - 1)


In [ ]:
%%writefile models/dreamer/rssm.py
import torch
import torch.nn as nn
import torch.nn.functional as F

class Encoder(nn.Module):
    """Encodes the 2-channel 64x64 grid (visible frame + memory map) into a dense representation."""
    def __init__(self, embed_dim=256):
        super().__init__()
        # Input: (B, 2, 64, 64)
        self.net = nn.Sequential(
            nn.Conv2d(2, 16, kernel_size=4, stride=2, padding=1), # (16, 32, 32)
            nn.SiLU(),
            nn.Conv2d(16, 32, kernel_size=4, stride=2, padding=1), # (32, 16, 16)
            nn.SiLU(),
            nn.Conv2d(32, 64, kernel_size=4, stride=2, padding=1), # (64, 8, 8)
            nn.SiLU(),
            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1), # (128, 4, 4)
            nn.SiLU(),
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, embed_dim),
            nn.LayerNorm(embed_dim)
        )
        
    def forward(self, obs):
        # obs is a dictionary of tensors, or we expect a tensor
        # Here we expect a tensor of shape (B, 2, 64, 64)
        return self.net(obs)

class Decoder(nn.Module):
    """Reconstructs the 2-channel 64x64 grid from the latent state."""
    def __init__(self, embed_dim=256):
        super().__init__()
        self.fc = nn.Linear(embed_dim, 128 * 4 * 4)
        self.net = nn.Sequential(
            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1), # (64, 8, 8)
            nn.SiLU(),
            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1), # (32, 16, 16)
            nn.SiLU(),
            nn.ConvTranspose2d(32, 16, kernel_size=4, stride=2, padding=1), # (16, 32, 32)
            nn.SiLU(),
            nn.ConvTranspose2d(16, 2, kernel_size=4, stride=2, padding=1), # (2, 64, 64)
        )
        
    def forward(self, latent):
        x = self.fc(latent)
        x = x.view(-1, 128, 4, 4)
        return self.net(x)

class RSSM(nn.Module):
    """
    Recurrent State-Space Model.
    For simplicity in this lightweight version, we use a continuous latent state
    instead of DreamerV3's discrete categorical latents, combined with a GRU.
    """
    def __init__(self, action_dim=5, deter_dim=256, stoch_dim=32, hidden_dim=256):
        super().__init__()
        self.deter_dim = deter_dim
        self.stoch_dim = stoch_dim
        
        # RNN for deterministic state
        self.cell = nn.GRUCell(hidden_dim, deter_dim)
        
        # Prior predictor: p(z_t | h_t)
        self.prior_net = nn.Sequential(
            nn.Linear(deter_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, 2 * stoch_dim) # mean and logvar
        )
        
        # Posterior predictor: q(z_t | h_t, x_t)
        self.post_net = nn.Sequential(
            nn.Linear(deter_dim + hidden_dim, hidden_dim), # deter_dim + embed_dim
            nn.SiLU(),
            nn.Linear(hidden_dim, 2 * stoch_dim)
        )
        
        # Input to RNN: h_t = f(h_{t-1}, z_{t-1}, a_{t-1})
        self.rnn_input_net = nn.Sequential(
            nn.Linear(stoch_dim + action_dim, hidden_dim),
            nn.SiLU()
        )
        
        # Predictors from state (h_t, z_t)
        state_dim = deter_dim + stoch_dim
        
        self.reward_net = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, 1) # predict symlog reward
        )
        
        self.continue_net = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, 1) # predict logit for termination
        )
        
    def initial(self, batch_size, device):
        return (
            torch.zeros(batch_size, self.deter_dim, device=device),
            torch.zeros(batch_size, self.stoch_dim, device=device)
        )
        
    def observe_step(self, prev_state, prev_action, embed):
        """One step of posterior (training)."""
        prev_deter, prev_stoch = prev_state
        
        # 1. Update deterministic state
        rnn_in = self.rnn_input_net(torch.cat([prev_stoch, prev_action], dim=-1))
        deter = self.cell(rnn_in, prev_deter)
        
        # 2. Prior
        prior_stats = self.prior_net(deter)
        prior_mean, prior_logvar = torch.chunk(prior_stats, 2, dim=-1)
        
        # 3. Posterior (uses observation embed)
        post_stats = self.post_net(torch.cat([deter, embed], dim=-1))
        post_mean, post_logvar = torch.chunk(post_stats, 2, dim=-1)
        
        # Sample stochastic state
        std = torch.exp(0.5 * post_logvar)
        stoch = post_mean + std * torch.randn_like(std)
        
        return (deter, stoch), (prior_mean, prior_logvar), (post_mean, post_logvar)
        
    def imagine_step(self, prev_state, prev_action):
        """One step of prior (imagination/rollout)."""
        prev_deter, prev_stoch = prev_state
        
        rnn_in = self.rnn_input_net(torch.cat([prev_stoch, prev_action], dim=-1))
        deter = self.cell(rnn_in, prev_deter)
        
        prior_stats = self.prior_net(deter)
        prior_mean, prior_logvar = torch.chunk(prior_stats, 2, dim=-1)
        
        std = torch.exp(0.5 * prior_logvar)
        stoch = prior_mean + std * torch.randn_like(std)
        
        return (deter, stoch)

class WorldModel(nn.Module):
    def __init__(self, action_dim=5):
        super().__init__()
        self.encoder = Encoder(embed_dim=256)
        self.decoder = Decoder(embed_dim=256+32) # decodes from state (deter + stoch)
        self.rssm = RSSM(action_dim=action_dim, deter_dim=256, stoch_dim=32, hidden_dim=256)


In [ ]:
%%writefile models/dreamer/actor_critic.py
import torch
import torch.nn as nn
from torch.distributions import Categorical

class ActorCritic(nn.Module):
    def __init__(self, state_dim=256+32, action_dim=5, hidden_dim=256):
        super().__init__()
        
        # Actor network: takes state (h_t, z_t) -> policy logits
        self.actor = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, action_dim)
        )
        
        # Critic network: takes state (h_t, z_t) -> state value (symlog)
        self.critic = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, 1)
        )
        
    def get_action_dist(self, state):
        """Returns a Categorical distribution over actions."""
        logits = self.actor(state)
        return Categorical(logits=logits)
        
    def get_value(self, state):
        """Returns the predicted value (in symlog space)."""
        return self.critic(state)


In [ ]:
%%writefile models/dreamer/planner.py
import torch
import torch.nn.functional as F
from models.dreamer.symlog import symexp

class LatentPlanner:
    def __init__(self, horizon=5, num_samples=100, action_dim=5, cem_iters=3, elite_frac=0.1):
        self.horizon = horizon
        self.num_samples = num_samples
        self.action_dim = action_dim
        self.cem_iters = cem_iters
        self.num_elites = max(1, int(num_samples * elite_frac))

    def plan(self, world_model, actor_critic, state, device):
        deter, stoch = state
        
        # We maintain a probability distribution over actions for each step in the horizon
        # Initial distribution is uniform over the 5 actions
        action_probs = torch.ones((self.horizon, self.action_dim), device=device) / self.action_dim
        
        best_action_ever = None
        best_return_ever = -float('inf')
        
        for i in range(self.cem_iters):
            # Sample action sequences for the whole horizon
            actions_dist = torch.distributions.Categorical(probs=action_probs)
            actions = actions_dist.sample((self.num_samples,)).T # Shape: (horizon, num_samples)
            
            # Mix in Actor policy for guidance (first 20 samples on first iteration)
            if i == 0:
                curr_deter = deter.repeat(20, 1)
                curr_stoch = stoch.repeat(20, 1)
                curr_state = (curr_deter, curr_stoch)
                for t in range(self.horizon):
                    full_state = torch.cat([curr_state[0], curr_state[1]], dim=-1)
                    a = actor_critic.get_action_dist(full_state).sample()
                    actions[t, :20] = a
                    curr_state = world_model.rssm.imagine_step(curr_state, F.one_hot(a, num_classes=self.action_dim).float())

            # Parallel Rollout
            curr_deter = deter.repeat(self.num_samples, 1)
            curr_stoch = stoch.repeat(self.num_samples, 1)
            curr_state = (curr_deter, curr_stoch)
            
            cumulative_rewards = torch.zeros(self.num_samples, device=device)
            first_actions = actions[0]
            
            for t in range(self.horizon):
                action_one_hot = F.one_hot(actions[t].long(), num_classes=self.action_dim).float()
                curr_state = world_model.rssm.imagine_step(curr_state, action_one_hot)
                
                next_full_state = torch.cat([curr_state[0], curr_state[1]], dim=-1)
                reward_pred = world_model.rssm.reward_net(next_full_state).squeeze(-1)
                cumulative_rewards += symexp(reward_pred)
                
            # Critic evaluation at end of horizon
            final_full_state = torch.cat([curr_state[0], curr_state[1]], dim=-1)
            value_pred = actor_critic.get_value(final_full_state).squeeze(-1)
            total_returns = cumulative_rewards + (0.99 ** self.horizon) * symexp(value_pred)
            
            # Find elites (top performers)
            elite_returns, elite_indices = torch.topk(total_returns, self.num_elites)
            elite_actions = actions[:, elite_indices] # shape (horizon, num_elites)
            
            # Update overall best
            if elite_returns[0].item() > best_return_ever:
                best_return_ever = elite_returns[0].item()
                best_action_ever = elite_actions[0, 0]
                
            # Refit distribution based on elites
            new_action_probs = torch.zeros_like(action_probs)
            for t in range(self.horizon):
                counts = torch.bincount(elite_actions[t].long(), minlength=self.action_dim).float()
                new_action_probs[t] = counts / self.num_elites
                
            # Smooth update
            action_probs = 0.8 * new_action_probs + 0.2 * action_probs
            
        return best_action_ever, best_return_ever


In [ ]:
%%writefile ls20_dreamer_env.py
import gymnasium as gym
from gymnasium import spaces
import numpy as np
import arc_agi
from arcengine import GameAction, GameState

class LS20DreamerEnv(gym.Env):
    """
    A Gymnasium wrapper for the ARC-AGI-3 ls20 environment.
    Includes Neural Map memory logic and reward shaping.
    """
    metadata = {"render_modes": ["human", "rgb_array"], "render_fps": 10}

    def __init__(self, render_mode=None):
        self.render_mode = render_mode
        import os
        import arc_agi
        self.arc = arc_agi.Arcade(
            environments_dir="/kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files",
            operation_mode=arc_agi.OperationMode.OFFLINE
        )
        
        # We start with the arc environment
        # we don't pass render_mode to arc.make because we will handle rendering of memory if needed
        self.env = self.arc.make("ls20")
        
        self.grid_size = 64
        self.num_colors = 16
        
        # Actions: 0=UP, 1=DOWN, 2=LEFT, 3=RIGHT, 4=ACTION1 (or ACTION5 depending on mapping)
        # In arcengine, actions are ACTION1, ACTION2, ACTION3, ACTION4, ACTION5, ACTION6
        # Typically 1-4 are movements, 5 is action, 6 is coordinate. 
        # We'll map discrete 0-4 to ACTION1-ACTION5.
        self.action_space = spaces.Discrete(5)
        self.action_mapping = [
            GameAction.ACTION1,
            GameAction.ACTION2,
            GameAction.ACTION3,
            GameAction.ACTION4,
            GameAction.ACTION5
        ]

        # Observation Space Dictionary
        # visible_frame: The raw frame (0 is assumed black/unseen)
        # memory_map: The aggregated memory
        # available_actions: One-hot vector of available actions
        self.observation_space = spaces.Dict({
            "visible_frame": spaces.Box(low=0, high=15, shape=(self.grid_size, self.grid_size), dtype=np.int8),
            "memory_map": spaces.Box(low=0, high=15, shape=(self.grid_size, self.grid_size), dtype=np.int8),
            "available_actions": spaces.MultiBinary(5)
        })

        # Internal state
        self.neural_map = np.zeros((self.grid_size, self.grid_size), dtype=np.int8)
        self.lives = 3
        self.current_level = 0
        self.last_frame_state = None
        self.last_visible_frame = np.zeros((self.grid_size, self.grid_size), dtype=np.int8)

    def _get_obs(self, frame_data):
        # Extract 64x64 frame
        frame = np.array(frame_data.frame[-1], dtype=np.int8)
        
        # Neural Map Aggregation
        # Assume 0 is the "dark/unseen" color. If 0 is meaningful, we might need a different heuristic,
        # but typically in ARC 0 is black background.
        visible_mask = (frame != 0)
        self.neural_map[visible_mask] = frame[visible_mask]

        # Available actions mask
        avail_mask = np.zeros(5, dtype=np.int8)
        for a in frame_data.available_actions:
            # a is 1 to 6
            idx = a - 1
            if 0 <= idx < 5:
                avail_mask[idx] = 1

        return {
            "visible_frame": frame,
            "memory_map": self.neural_map.copy(),
            "available_actions": avail_mask
        }

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        
        # Reset the underlying env
        self.env = self.arc.make("ls20")
        frame_data = self.env.reset()
        
        # Reset internal tracking
        self.neural_map = np.zeros((self.grid_size, self.grid_size), dtype=np.int8)
        self.lives = 3
        self.current_level = frame_data.levels_completed
        self.last_frame_state = frame_data.state
        
        obs = self._get_obs(frame_data)
        self.last_visible_frame = obs["visible_frame"].copy()
        
        info = {"lives": self.lives, "level": self.current_level}
        
        return obs, info

    def step(self, action_idx):
        action = self.action_mapping[action_idx]
        frame_data = self.env.step(action)
        
        if frame_data is None:
            # End of everything
            return self._get_empty_obs(), 0, True, False, {}

        # 1. Base Step Penalty
        reward = -0.01
        terminated = False
        
        # 2. Level Check
        new_level = frame_data.levels_completed
        if new_level > self.current_level:
            reward += 10.0
            self.current_level = new_level
            # Also clear the memory map for the new level
            self.neural_map = np.zeros((self.grid_size, self.grid_size), dtype=np.int8)
            
        # 3. Game Over / Full Reset Check (Fuel ran out -> lost a life)
        if frame_data.full_reset or frame_data.state == GameState.GAME_OVER:
            # The agent died (ran out of fuel or hit a trap)
            self.lives -= 1
            reward -= 10.0
            # Reset memory map because it respawns or resets level
            self.neural_map = np.zeros((self.grid_size, self.grid_size), dtype=np.int8)
            
            if self.lives <= 0:
                terminated = True
            elif frame_data.state == GameState.GAME_OVER:
                # Intercept the native GAME_OVER and use one of our lives
                frame_data = self.env.reset()
                self.current_level = frame_data.levels_completed
                
        # 4. Win Check
        if frame_data.state == GameState.WIN:
            reward += 10.0
            terminated = True
            
        self.last_frame_state = frame_data.state
        
        obs = self._get_obs(frame_data)
        
        # 5. Intrinsic Motivation: Reward for changing the frame (moving instead of hitting walls)
        if not np.array_equal(self.last_visible_frame, obs["visible_frame"]):
            reward += 0.1
        else:
            reward -= 0.1  # Extra penalty for standing still / hitting a wall
            
        self.last_visible_frame = obs["visible_frame"].copy()
        
        info = {"lives": self.lives, "level": self.current_level}
        
        return obs, reward, terminated, False, info

    def _get_empty_obs(self):
        return {
            "visible_frame": np.zeros((self.grid_size, self.grid_size), dtype=np.int8),
            "memory_map": np.zeros((self.grid_size, self.grid_size), dtype=np.int8),
            "available_actions": np.zeros(5, dtype=np.int8)
        }

    def render(self):
        if self.render_mode == "human":
            pass # We will handle rendering externally in play_and_learn or implement PyGame here

    def close(self):
        pass



In [ ]:
%%writefile play_and_learn.py
import os
# --- CRITICAL: Set these BEFORE any other imports ---
# This ensures the arc_agi library sees the path during initialization
os.environ["ARC_GAMES_DIR"] = "/kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files"
os.environ["OPERATION_MODE"] = "offline"
os.environ["ARC_API_KEY"] = "test-key-123"

import argparse
import numpy as np
import torch
import torch.nn.functional as F
from collections import deque
import random
import matplotlib.pyplot as plt

from ls20_dreamer_env import LS20DreamerEnv
from models.dreamer.rssm import WorldModel
from models.dreamer.actor_critic import ActorCritic
from models.dreamer.symlog import symlog, symexp
from models.dreamer.planner import LatentPlanner

class ReplayBuffer:
    def __init__(self, capacity=10000):
        self.buffer = deque(maxlen=capacity)
        
    def add(self, obs, action, reward, done):
        self.buffer.append((obs, action, reward, done))
        
    def sample(self, batch_size, sequence_length):
        valid_indices = [i for i in range(len(self.buffer) - sequence_length) 
                         if not any(self.buffer[i+j][3] for j in range(sequence_length-1))]
        if len(valid_indices) < batch_size:
            indices = np.random.choice(len(self.buffer) - sequence_length, batch_size)
        else:
            indices = np.random.choice(valid_indices, batch_size)
        
        obs_seq, action_seq, reward_seq = [], [], []
        for i in indices:
            o_seq, a_seq, r_seq = [], [], []
            for j in range(sequence_length):
                o, a, r, d = self.buffer[i+j]
                o_seq.append(o)
                a_seq.append(a)
                r_seq.append(r)
            obs_seq.append(o_seq)
            action_seq.append(a_seq)
            reward_seq.append(r_seq)
            
        return obs_seq, action_seq, reward_seq
        
    def __len__(self):
        return len(self.buffer)

def preprocess_obs(obs, device):
    """Stack visible_frame and memory_map into a 2-channel tensor."""
    vf = torch.tensor(obs["visible_frame"], dtype=torch.float32)
    mm = torch.tensor(obs["memory_map"], dtype=torch.float32)
    # Scale 0-15 to 0-1
    vf = vf / 15.0
    mm = mm / 15.0
    stacked = torch.stack([vf, mm], dim=0).unsqueeze(0).to(device) # (1, 2, 64, 64)
    return stacked

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--render_memory", action="store_true", help="Visualize the Neural Map memory.")
    args = parser.parse_args()

    device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
    print(f"Using device: {device}")

    env = LS20DreamerEnv()
    
    world_model = WorldModel(action_dim=5).to(device)
    actor_critic = ActorCritic(state_dim=256+32, action_dim=5, hidden_dim=256).to(device)
    
    # Auto-resume from latest checkpoint if available
    import glob
    wm_checkpoints = glob.glob("checkpoints/world_model_step_*.pth")
    if wm_checkpoints:
        wm_checkpoints.sort(key=lambda x: int(x.split('_step_')[1].split('.pth')[0]))
        latest_wm = wm_checkpoints[-1]
        latest_ac = latest_wm.replace("world_model", "actor_critic")
        if os.path.exists(latest_ac):
            print(f"🔄 Resuming from checkpoint: {latest_wm}")
            world_model.load_state_dict(torch.load(latest_wm, map_location=device))
            actor_critic.load_state_dict(torch.load(latest_ac, map_location=device))
            
    target_actor_critic = ActorCritic(state_dim=256+32, action_dim=5, hidden_dim=256).to(device)
    target_actor_critic.load_state_dict(actor_critic.state_dict())
    target_actor_critic.eval()
    
    wm_optimizer = torch.optim.Adam(world_model.parameters(), lr=1e-4)
    ac_optimizer = torch.optim.Adam(actor_critic.parameters(), lr=3e-5)
    
    buffer = ReplayBuffer(capacity=50000)
    
    obs, info = env.reset()
    
    # Initialize RSSM state
    state = world_model.rssm.initial(1, device)
    
    if args.render_memory:
        plt.ion()
        fig, axes = plt.subplots(1, 2, figsize=(10, 5))
        img_vis = axes[0].imshow(obs["visible_frame"], vmin=0, vmax=15, cmap="tab20")
        axes[0].set_title("Visible Frame (Arc-AGI)")
        img_mem = axes[1].imshow(obs["memory_map"], vmin=0, vmax=15, cmap="tab20")
        axes[1].set_title("Neural Map (Memory)")
        plt.show()

    steps = 0
    episode_reward = 0
    current_lives = info.get("lives", 3)
    current_level = info.get("level", 0)
    
    # Initialize Test-Time Planner
    planner = LatentPlanner(horizon=5, num_samples=100)
    
    print("Starting online Play & Learn loop...")
    
    while True:
        steps += 1
        
        # 1. Play: Choose action using Actor
        with torch.no_grad():
            obs_tensor = preprocess_obs(obs, device)
            embed = world_model.encoder(obs_tensor)
            
            # Since we are online, we don't know the previous action without tracking, 
            # we'll pass a dummy zero action for the first step, or keep track.
            prev_action_tensor = torch.zeros(1, 5, device=device) # simplified for prototype
            
            # Update state with posterior (using actual observation)
            state, prior, post = world_model.rssm.observe_step(state, prev_action_tensor, embed)
            
            # Concatenate deter and stoch to get full state
            deter, stoch = state
            full_state = torch.cat([deter, stoch], dim=-1)
            
            # Get action distribution for logging
            action_dist = actor_critic.get_action_dist(full_state)
            
            # Thinking Step: Use LatentPlanner to find the best action
            action, expected_return = planner.plan(world_model, actor_critic, state, device)
            
            # Print action debugging occasionally
            if steps % 50 == 0:
                probs = F.softmax(action_dist.logits, dim=-1).cpu().numpy()[0]
                print(f"[DEBUG] Step {steps} | Level: {current_level} | Expected Return: {expected_return:.2f} | Selected: {action.item()}")
                
        action_val = action.item()
        
        # 2. Step in environment
        next_obs, reward, terminated, truncated, info = env.step(action_val)
        episode_reward += reward
        
        # Check for level up
        if info["level"] > current_level:
            print(f"[{steps}] 🌟 LEVEL UP! Advanced to Level {info['level']}")
            current_level = info["level"]
            
        # Check for life loss based on info
        if info["lives"] < current_lives:
            current_lives = info["lives"]
            if current_lives > 0:
                print(f"[{steps}] ⚠️  FUEL DEPLETED! Agent died. Lives remaining: {current_lives}/3")
            else:
                print(f"[{steps}] 💀  GAME OVER! All fuel/lives exhausted.")
        
        # 3. Render Memory if requested
        if args.render_memory and steps % 2 == 0:
            img_vis.set_data(next_obs["visible_frame"])
            img_mem.set_data(next_obs["memory_map"])
            plt.pause(0.01)
            
        # 4. Add to buffer
        buffer.add(obs, action_val, reward, terminated)
        obs = next_obs
        
        if terminated:
            print(f"Step {steps} | Environment Reset | Episode Reward: {episode_reward:.2f} | Level: {info['level']}")
            obs, info = env.reset()
            current_lives = info.get("lives", 3)
            current_level = info.get("level", 0)
            state = world_model.rssm.initial(1, device)
            episode_reward = 0
            
        # 5. Train (Online Updates)
        if len(buffer) > 64 and steps % 5 == 0:
            seq_len = 16
            batch_size = 32
            if len(buffer) < seq_len + 1:
                continue
                
            wm_optimizer.zero_grad()
            o_seq, a_seq, r_seq = buffer.sample(batch_size, seq_len)
            
            b_state = world_model.rssm.initial(batch_size, device)
            prev_action = torch.zeros(batch_size, 5, device=device)
            
            post_states = []
            reward_loss = 0
            kl_loss = 0
            
            # Backpropagation Through Time (BPTT)
            for t in range(seq_len):
                o_t = torch.cat([preprocess_obs(o_seq[b][t], device) for b in range(batch_size)], dim=0)
                a_t = F.one_hot(torch.tensor([a_seq[b][t] for b in range(batch_size)], device=device).long(), num_classes=5).float()
                r_t = torch.tensor([r_seq[b][t] for b in range(batch_size)], dtype=torch.float32, device=device).unsqueeze(1)
                
                embeds = world_model.encoder(o_t)
                b_state, b_prior, b_post = world_model.rssm.observe_step(b_state, prev_action, embeds)
                prev_action = a_t
                
                b_full_state = torch.cat([b_state[0], b_state[1]], dim=-1)
                post_states.append(b_full_state)
                
                pred_reward = world_model.rssm.reward_net(b_full_state)
                reward_loss = reward_loss + F.mse_loss(pred_reward, symlog(r_t))
                
                prior_mean, prior_logvar = b_prior
                post_mean, post_logvar = b_post
                kl_step = -0.5 * torch.sum(1 + post_logvar - prior_logvar - ((post_mean - prior_mean).pow(2) + post_logvar.exp()) / prior_logvar.exp(), dim=1).mean()
                kl_loss = kl_loss + kl_step
                
            wm_loss = (reward_loss + 0.1 * kl_loss) / seq_len
            wm_loss.backward()
            torch.nn.utils.clip_grad_norm_(world_model.parameters(), 100.0)
            wm_optimizer.step()
            
            # --- Actor Critic Update (Imagination) ---
            ac_optimizer.zero_grad()
            
            # Flatten post_states to imagine from all visited states
            flattened_states = torch.cat(post_states, dim=0).detach()
            deter_dim = 256
            curr_state = (flattened_states[:, :deter_dim], flattened_states[:, deter_dim:])
            
            H = 15
            img_rewards = []
            img_values = []
            img_log_probs = []
            img_entropies = []
            
            # Imagine trajectories
            for t in range(H):
                full_s = torch.cat([curr_state[0], curr_state[1]], dim=-1)
                dist = actor_critic.get_action_dist(full_s)
                action = dist.sample()
                
                img_log_probs.append(dist.log_prob(action))
                img_entropies.append(dist.entropy())
                
                a_tensor = F.one_hot(action.long(), num_classes=5).float()
                curr_state = world_model.rssm.imagine_step(curr_state, a_tensor)
                next_full_s = torch.cat([curr_state[0], curr_state[1]], dim=-1)
                
                img_rewards.append(symexp(world_model.rssm.reward_net(next_full_s)))
                img_values.append(symexp(target_actor_critic.get_value(next_full_s)))
                
            # Compute Lambda Returns backward
            returns = img_values[-1].detach()
            actor_loss = 0
            
            for t in reversed(range(H)):
                returns = img_rewards[t].detach() + 0.99 * returns
                actor_loss = actor_loss - (img_log_probs[t] * symlog(returns).squeeze(-1)).mean() - 0.05 * img_entropies[t].mean()
                
            actor_loss = actor_loss / H
            
            # Critic Loss fits the very first state to the imagined return
            ret0 = img_rewards[0].detach() + 0.99 * img_values[0].detach()
            val_pred = actor_critic.get_value(flattened_states)
            critic_loss = F.mse_loss(val_pred, symlog(ret0))
            
            ac_loss = actor_loss + critic_loss
            ac_loss.backward()
            torch.nn.utils.clip_grad_norm_(actor_critic.parameters(), 100.0)
            ac_optimizer.step()
            
            # Polyak Averaging for Target Critic
            with torch.no_grad():
                for param, target_param in zip(actor_critic.parameters(), target_actor_critic.parameters()):
                    target_param.data.copy_(0.01 * param.data + 0.99 * target_param.data)
            
            if steps % 100 == 0:
                print(f"[DEBUG] Loss | WM: {wm_loss.item():.4f} (Reward: {(reward_loss/seq_len).item():.4f}, KL: {(kl_loss/seq_len).item():.4f}) | AC: {ac_loss.item():.4f} (Actor: {actor_loss.item():.4f}, Critic: {critic_loss.item():.4f})")
                
            # Save Checkpoints
            if steps % 5000 == 0:
                os.makedirs("checkpoints", exist_ok=True)
                torch.save(world_model.state_dict(), f"checkpoints/world_model_step_{steps}.pth")
                torch.save(actor_critic.state_dict(), f"checkpoints/actor_critic_step_{steps}.pth")
                print(f"[{steps}] 💾 Model checkpoints saved to disk!")

if __name__ == "__main__":
    main()


# !python play_and_learn.py 2>&1 | tee /kaggle/working/logs/training.log # keep this here. 

In [ ]:
import os

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    # Wait for the ARC gateway to initialize
    !curl --fail --retry 99 --retry-all-errors --retry-delay 5 --retry-max-time 600 http://gateway:8001/api/games
    
    # Configure arc-agi to point to the local Kaggle gateway
    with open('.env', 'w') as f:
        f.write("""SCHEME=http
HOST=gateway
PORT=8001
ARC_API_KEY=test-key-123
ARC_BASE_URL=http://gateway:8001/
OPERATION_MODE=online
RECORDINGS_DIR=/kaggle/working/server_recording
""")

    # Run the Dreamer Agent
    !MPLBACKEND=agg python play_and_learn.py
else:
    # Fast fallback for saving the notebook version
    import pandas as pd
    submission = pd.DataFrame(data=[['1_0', '1', True, 1]], columns=['row_id', 'game_id', 'end_of_game', 'score'])
    submission.to_parquet('/kaggle/working/submission.parquet', index=False)
